In [58]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import re
import networkx as nx
import plotly.graph_objects as go

from typing import Any, Iterable, Optional

from IPython.display import display, Markdown
from dotenv import load_dotenv
from openai import OpenAI, BadRequestError
from pyvis.network import Network

import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "working"))
from event_graph_cleaner import clean_graph


load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise RuntimeError("OPENROUTER_API_KEY not found. Add it to your .env file.")

# OpenRouter is OpenAI-API compatible: same SDK, different base_url + model slugs
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# Optional headers used by OpenRouter for attribution / rankings
OPENROUTER_HEADERS = {
    "HTTP-Referer": "http://localhost",
    "X-Title": "Supply Chain Study",
}


# OpenRouter model
model_lists = [
    "~openai/gpt-mini-latest", 
    "google/gemini-3.5-flash-lite",                
    "anthropic/claude-haiku-4.5"
    ]

MODEL = model_lists[2]
JUDGE_MODEL_NAME = "google/gemini-3.5-flash-lite" # "openai/gpt-5"
# MODEL_NAME = "openai/gpt-5.6-luna" # "openai/gpt-5-mini"


start_date = "2023-07-11"
end_date = "2023-07-17"
top_n = 20

print("✓ Imports complete")
print(f"Selected Model: {MODEL}\nLLM as Judge Model: {JUDGE_MODEL_NAME}")


✓ Imports complete
Selected Model: anthropic/claude-haiku-4.5
LLM as Judge Model: google/gemini-3.5-flash-lite


In [11]:
feature_importance = pd.read_csv(f"../../feature_importance//feature_importance_{start_date}_{end_date}/feature_importance_from_{start_date}_to_{end_date}_top_{top_n}.csv")
causality = pd.read_csv(f"../../causality/causality_{start_date}_{end_date}/causality_from_{start_date}_to_{end_date}_top_{top_n}.csv")
merged_graph = pd.read_csv(f"../../merged_graph/merged_graph_{end_date}_feature_importance_top_{top_n}.csv")
taxonomy = pd.read_csv("../../taxonomy/supply_chain_taxonomy_google_gemini-3.5-flash-lite.csv")

macroeconomic_features = ['interest_rate_value', 'stock_market_close', 'inflation_rate', 'balance_of_trade', 'exports', 'imports', 'currency_log_return', 'currency_momentum_5d']


In [12]:
display(feature_importance)
display(causality)

,Feature name,feature importance,shap_signed,direction
0,Delivery,755.291614,30.242991,positive
1,delivery_sales_ratio,555.077020,-101.236309,negative
2,Production,162.908200,-29.394634,negative
3,production_sales_ratio,148.768812,-13.426749,negative
4,Factory_Issue,130.705022,-14.828466,negative
5,inventory_proxy,87.777145,23.566890,positive
6,Sales_lag_21,58.068483,9.020391,positive
7,Sales_lag_1,49.230745,-7.776972,negative
8,Sales_lag_2,47.428629,-25.431382,negative
9,Sales_lag_28,43.166130,-3.899125,negative


,cause,effect,lag,coefficient,direction
0,Sales_lag_28,Production,0.0,0.932276,positive
1,delivery_sales_ratio,production_sales_ratio,0.0,0.909791,positive
2,Sales_lag_28,Sales_lag_3,0.0,0.850840,positive
3,Sales_lag_28,Sales_lag_7,0.0,0.834155,positive
4,Sales_lag_7,Sales_lag_21,0.0,0.747857,positive
...,...,...,...,...,...
103,Product_SOS001L12P,Delivery,1.0,-0.055638,negative
104,Sales_lag_14,Sales_lag_1,1.0,-0.055528,negative
105,Product_SOS001L12P,Factory_Issue,1.0,-0.055279,negative
106,Product_SOS001L12P,Sales,1.0,-0.050122,negative


In [13]:
# Check how many macroeconomic features exist in feature_importance["Feature name"]
existing_macro = [f for f in macroeconomic_features if f in feature_importance["Feature name"].values]
missing_macro = [f for f in macroeconomic_features if f not in feature_importance["Feature name"].values]

print(f"Found {len(existing_macro)} of {len(macroeconomic_features)} macroeconomic features in feature_importance:")
for f in existing_macro:
    print(f"  ✓ {f}")

if missing_macro:
    print(f"\nMissing {len(missing_macro)} features:")
    for f in missing_macro:
        print(f"  ✗ {f}")

Found 3 of 8 macroeconomic features in feature_importance:
  ✓ stock_market_close
  ✓ balance_of_trade
  ✓ exports

Missing 5 features:
  ✗ interest_rate_value
  ✗ inflation_rate
  ✗ imports
  ✗ currency_log_return
  ✗ currency_momentum_5d


In [14]:
cols = ["edge_type", "source_node", ""]
merged_graph.drop(columns=["Unnamed: 0"])

,edge_type,source_node,source_type,target_node,target_type,edge_label,feature,feature_in_causality_graph,lag,coefficient,...,evidence_quote,scope_justification,quote_similarity,quote_best_match,judge_verdict,judge_reasoning,article_source,article_date,article_url,article_body_preview
0,causality,Sales_lag_28,feature,Production,feature,positive,Sales_lag_28,True,0.0,0.932276,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,causality,delivery_sales_ratio,feature,production_sales_ratio,feature,positive,delivery_sales_ratio,True,0.0,0.909791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,causality,Sales_lag_28,feature,Sales_lag_3,feature,positive,Sales_lag_28,True,0.0,0.850840,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,causality,Sales_lag_28,feature,Sales_lag_7,feature,positive,Sales_lag_28,True,0.0,0.834155,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,causality,Sales_lag_7,feature,Sales_lag_21,feature,positive,Sales_lag_7,True,0.0,0.747857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,event,inflation_rate,feature,Crises raise prices\n(2022-12-28),event,raises price levels,inflation_rate,False,NaN,NaN,...,The Covid-19 pandemic and the Russia-Ukraine c...,The article attributes broad increases across ...,0.993,"""The Covid-19 pandemic and the Russia-Ukraine ...",CONFIRMED,The article explicitly states that the Covid-1...,MSN International Edition,2022-12-28,https://www.msn.com/en-xl/asia/bangladesh/priv...,"Dhaka, Dec. 28 -- FBCCI president Md Jashim Ud..."
572,event,inflation_rate,feature,Proposed budget targets inflation\n(2023-06-02),event,aims to contain,inflation_rate,False,NaN,NaN,...,mainly aimed to contain inflation and maintain...,The article links the national budget to conta...,0.993,mainly aimed to contain inflation and maintain...,CONFIRMED,The article explicitly states the budget is 'm...,United News of India,2023-06-02,http://www.uniindia.com/news/world/b-desh-fina...,"Dhaka, June 2 (UNI) Bangladesh Finance Ministe..."
573,event,inflation_rate,feature,Consumer prices raise inflation\n(2023-04-05),event,raises inflation,inflation_rate,False,NaN,NaN,...,Inflation in Bangladesh jumped to 9.33 percent...,The article explicitly describes a country-wid...,0.995,Inflation in Bangladesh jumped to 9.33 percent...,CONFIRMED,The article states that Bangladesh’s inflation...,english.news.cn,2023-04-05,https://english.news.cn/asiapacific/20230405/a...,"DHAKA, April 5 (Xinhua) -- Inflation in Bangla..."
574,event,exports,feature,"Policy raises exports, aids recovery\n(2022-09...",event,supports economic recovery,exports,True,NaN,NaN,...,the government's policy of raising exports and...,The article explicitly describes a national go...,0.995,the government's policy of raising exports and...,CONFIRMED,"The article states, “the government's policy o...",LatestLY,2022-09-10,https://www.latestly.com/agency-news/world-new...,"Dhaka [Bangladesh], September 10 (ANI): Econom..."


feature -> edge_label -> event summary

| Position | Value                                  | Role                                                                                                                                                                               |
| -------- | -------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1st      | "exports"                              | Head entity / subject — the entity being acted upon or discussed                                                                                                                   |
| 2nd      | "reduces export growth"                | Relation / predicate — the causal or semantic link connecting subject to object; here it's a causal-effect phrase rather than a simple verb                                        |
| 3rd      | "Customer cost pressures slow exports" | Tail entity / object, or source sentence / evidence span — likely the originating text snippet the triplet was extracted from, or the "cause" entity in a (cause → effect) framing |

In [15]:
feature_importance

,Feature name,feature importance,shap_signed,direction
0,Delivery,755.291614,30.242991,positive
1,delivery_sales_ratio,555.077020,-101.236309,negative
2,Production,162.908200,-29.394634,negative
3,production_sales_ratio,148.768812,-13.426749,negative
4,Factory_Issue,130.705022,-14.828466,negative
5,inventory_proxy,87.777145,23.566890,positive
6,Sales_lag_21,58.068483,9.020391,positive
7,Sales_lag_1,49.230745,-7.776972,negative
8,Sales_lag_2,47.428629,-25.431382,negative
9,Sales_lag_28,43.166130,-3.899125,negative


feature_importance is the mean absolute SHAP value across all predictions (i.e., mean(|SHAP_i|)), which measures the average magnitude of a feature's impact on the model's output regardless of direction. This is the standard way SHAP ranks global feature importance, and it's what you should use to answer "how important is this feature overall."

In [16]:
causality

,cause,effect,lag,coefficient,direction
0,Sales_lag_28,Production,0.0,0.932276,positive
1,delivery_sales_ratio,production_sales_ratio,0.0,0.909791,positive
2,Sales_lag_28,Sales_lag_3,0.0,0.850840,positive
3,Sales_lag_28,Sales_lag_7,0.0,0.834155,positive
4,Sales_lag_7,Sales_lag_21,0.0,0.747857,positive
...,...,...,...,...,...
103,Product_SOS001L12P,Delivery,1.0,-0.055638,negative
104,Sales_lag_14,Sales_lag_1,1.0,-0.055528,negative
105,Product_SOS001L12P,Factory_Issue,1.0,-0.055279,negative
106,Product_SOS001L12P,Sales,1.0,-0.050122,negative


In [17]:
taxonomy

,feature_name,child,level
0,Supply Chain Dataset,Calendar and Events,1
1,Supply Chain Dataset,External Factors,1
2,Supply Chain Dataset,Operations and Logistics,1
3,Supply Chain Dataset,Product Taxonomy,1
4,Calendar and Events,Category,2
...,...,...,...
137,Weather,Humidity,3
138,Weather,Rainfall,3
139,Weather,Sunshine,3
140,Weather,Temperature,3


<br> <br>

## Generate Explanation using Feature Importance

In [9]:
def build_parent_map(taxonomy_df: pd.DataFrame,
                     child_col: str = "child",
                     parent_col: str = "feature_name",
                     level_col: str = "level") -> tuple[dict[str, str], dict[str, int]]:
    """
    Build `child -> parent` and `node -> level` lookup dictionaries from a taxonomy dataframe.

    The taxonomy is expected to be an edge list: one row per (parent, child) pair.
    """
    parent_of: dict[str, str] = {}
    level_of: dict[str, int] = {}

    for _, row in taxonomy_df.iterrows():
        child = str(row[child_col]).strip()
        parent = str(row[parent_col]).strip()
        if not child or child.lower() == "nan":
            continue
        parent_of[child] = parent
        if level_col in taxonomy_df.columns and pd.notna(row[level_col]):
            level_of[child] = int(row[level_col])
        # parents that never appear as a child are roots (level 0)
        level_of.setdefault(parent, max(int(row[level_col]) - 1, 0) if level_col in taxonomy_df.columns and pd.notna(row[level_col]) else 0)

    return parent_of, level_of


def _resolve_node(feature: str, parent_of: dict[str, str]) -> Optional[str]:
    """
    Try to match a feature name to a node in the taxonomy.

    Falls back to stripping engineered suffixes such as `_lag_7`, `_roll_14`, `_ma_3`
    so that e.g. `Salary_lag_7` resolves to `Salary`.
    """
    if feature in parent_of:
        return feature

    candidates = [
        re.sub(r"_(lag|lead|roll|rolling|ma|mean|std|diff|pct|ratio)_?\d*$", "", feature, flags=re.IGNORECASE),
        feature.split("_lag_")[0],
        feature.rsplit("_", 1)[0],
    ]
    for cand in candidates:
        cand = cand.strip()
        if cand and cand != feature and cand in parent_of:
            return cand

    # case-insensitive last resort
    lower_map = {k.lower(): k for k in parent_of}
    if feature.lower() in lower_map:
        return lower_map[feature.lower()]

    return None


def trace_feature_hierarchy(features: pd.Series,
                            taxonomy_df: pd.DataFrame,
                            child_col: str = "child",
                            parent_col: str = "feature_name",
                            level_col: str = "level",
                            verbose: bool = True) -> pd.DataFrame:
    """
    For every feature in `features`, walk up the taxonomy until the highest-ranking
    (root-most) ancestor is reached.

    Parameters
    ----------
    features : pd.Series
        e.g. ``feature_importance["Feature name"]``
    taxonomy_df : pd.DataFrame
        Taxonomy edge list with `feature_name` (parent), `child` and `level` columns.
    verbose : bool
        When True, prints the hierarchy for each feature, e.g.
        ``Salary_lag_7 -> Salary -> Operations and Logistics -> Supply Chain Dataset``

    Returns
    -------
    pd.DataFrame
        Columns: feature, matched_node, immediate_parent, top_parent,
        highest_rank_level, depth, path, found
    """
    parent_of, level_of = build_parent_map(taxonomy_df, child_col, parent_col, level_col)

    records: list[dict[str, Any]] = []

    for feature in features.astype(str):
        node = _resolve_node(feature, parent_of)

        if node is None:
            path = [feature]
            records.append({
                "feature": feature,
                "matched_node": None,
                "immediate_parent": None,
                "top_parent": None,
                "highest_rank_level": None,
                "depth": 0,
                "path": feature,
                "found": False,
            })
            if verbose:
                print(f"{feature} -> [not found in taxonomy]")
            continue

        # path starts at the feature itself (keep the raw name if it differed)
        path = [feature] if node == feature else [feature, node]

        current = node
        seen = {current}
        while current in parent_of:
            parent = parent_of[current]
            if parent in seen:  # guard against cycles
                break
            path.append(parent)
            seen.add(parent)
            current = parent

        top_parent = path[-1]
        ancestors = [p for p in path[1:] if p != feature]

        records.append({
            "feature": feature,
            "matched_node": node,
            "immediate_parent": parent_of.get(node),
            "top_parent": top_parent,
            "highest_rank_level": level_of.get(top_parent, 0),
            "depth": len(ancestors),
            "path": " -> ".join(path),
            "found": True,
        })

        if verbose:
            print(" -> ".join(path))

    return pd.DataFrame.from_records(records)


hierarchy_df = trace_feature_hierarchy(feature_importance["Feature name"], taxonomy)
display(hierarchy_df)


Delivery -> Operations and Logistics -> Supply Chain Dataset
delivery_sales_ratio -> Operations and Logistics -> Supply Chain Dataset
Production -> Operations and Logistics -> Supply Chain Dataset
production_sales_ratio -> Operations and Logistics -> Supply Chain Dataset
Factory_Issue -> Operations and Logistics -> Supply Chain Dataset
inventory_proxy -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_21 -> Sales -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_1 -> Sales -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_2 -> Sales -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_28 -> Sales -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_7 -> Sales -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_3 -> Sales -> Operations and Logistics -> Supply Chain Dataset
Sales_lag_14 -> Sales -> Operations and Logistics -> Supply Chain Dataset
factory_issue_rate -> Operations and Logistics -> Supply Chain Dataset
Product_S

,feature,matched_node,immediate_parent,top_parent,highest_rank_level,depth,path,found
0,Delivery,Delivery,Operations and Logistics,Supply Chain Dataset,0,2,Delivery -> Operations and Logistics -> Supply...,True
1,delivery_sales_ratio,delivery_sales_ratio,Operations and Logistics,Supply Chain Dataset,0,2,delivery_sales_ratio -> Operations and Logisti...,True
2,Production,Production,Operations and Logistics,Supply Chain Dataset,0,2,Production -> Operations and Logistics -> Supp...,True
3,production_sales_ratio,production_sales_ratio,Operations and Logistics,Supply Chain Dataset,0,2,production_sales_ratio -> Operations and Logis...,True
4,Factory_Issue,Factory_Issue,Operations and Logistics,Supply Chain Dataset,0,2,Factory_Issue -> Operations and Logistics -> S...,True
5,inventory_proxy,inventory_proxy,Operations and Logistics,Supply Chain Dataset,0,2,inventory_proxy -> Operations and Logistics ->...,True
6,Sales_lag_21,Sales_lag_21,Sales,Supply Chain Dataset,0,3,Sales_lag_21 -> Sales -> Operations and Logist...,True
7,Sales_lag_1,Sales_lag_1,Sales,Supply Chain Dataset,0,3,Sales_lag_1 -> Sales -> Operations and Logisti...,True
8,Sales_lag_2,Sales_lag_2,Sales,Supply Chain Dataset,0,3,Sales_lag_2 -> Sales -> Operations and Logisti...,True
9,Sales_lag_28,Sales_lag_28,Sales,Supply Chain Dataset,0,3,Sales_lag_28 -> Sales -> Operations and Logist...,True


In [ ]:
cols = ["feature", "immediate_parent"]
hierarchy_df[cols]

In [ ]:
cols = ["feature", "immediate_parent"]

feature_impoprtance_for_explanation = feature_importance.merge(
    hierarchy_df[cols],
    how="left",
    left_on="Feature name",
    right_on="feature",
).drop(columns=["feature", "shap_signed"])

display(feature_impoprtance_for_explanation)


In [ ]:

# -----------------------------------------------------------------------------------------------------------------------------------
# ------------------------------------------------------------- Version 1 -----------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------------------------------



# # =============================================================================
# # PROMPT PLAYGROUND — edit freely, then re-run the generation cell below.
# # =============================================================================

# EXPLANATION_SYSTEM_PROMPT = """\
# You are a senior supply-chain data scientist writing the interpretation paragraph of a
# demand-forecasting report in Bangladesh.

# You are given SHAP-based global feature attributions from a trained sales-forecasting
# model, aggregated to the parent categories of a supply-chain taxonomy.

# Ground rules:
# - SHAP tells you ASSOCIATION, never causation. Never claim a category "causes" the outcome.
# - Use ONLY the numbers provided. Never invent categories, values, or external events.
# - `importance` = sum of mean(|SHAP|) over the category -> magnitude of influence.
# - Refer to drivers ONLY by their taxonomy category name (e.g. "Operations and Logistics",
#   "Sales", "Macroeconomics"). NEVER mention raw feature or column names such as
#   `delivery_sales_ratio`, `Sales_lag_7` or `Delivery`.

# Voice and style:
# - Write the way an experienced analyst actually writes for colleagues: calm, specific,
#   and substantive. Explain what the attributions mean for demand behaviour, not where
#   each category sits in a ranking.
# - Do NOT narrate the leaderboard. Never use rank-announcing or hype constructions. These
#   are forbidden, along with any close paraphrase: "dominates", "by a wide margin",
#   "the next most influential", "the next substantive contributor", "the next largest",
#   "second most", "comes in second", "ranks third", "the top driver", "followed by",
#   "the lion's share", "overwhelmingly", "far and away", "clearly leads",
#   "in comparison", "the remaining categories".
# - Never introduce a category by its position in the ordering. Introduce it by what it
#   represents or by what its attribution implies about demand, then attach the number.
# - Weave the numbers into sentences that carry an idea; never produce a sentence whose
#   only content is a category name plus a percentage.
# - Do not chain the small categories into a run of name-plus-percentage clauses; describe
#   their negligible contribution in your own words and cite at most a couple of figures.
# - Vary sentence structure. Avoid template-like enumeration ("First... Second... Third...").
# - No filler, no restating the task, no headings, no bullet lists, no tables, no hedging
#   boilerplate.
# """

# EXPLANATION_USER_PROMPT_TEMPLATE = """\
# ## Forecasting target
# {target}

# ## Taxonomy category attributions
# {category_table}

# ## Task
# Write ONE single paragraph of at most {max_words} words interpreting these attributions
# for a planning audience. The paragraph should make clear, as part of a natural argument
# rather than a ranked list:
# - where the model concentrates most of its explanatory weight, and what that says about
#   how demand behaves;
# - how the other materially contributing categories shape the forecast, including the
#   direction in which each pushes it;
# - which categories contribute so little that they can be set aside for now;
# - one concrete monitoring action a planner should take, tied to a named category;
# - a closing acknowledgement that these are associations rather than causes and that
#   historical-sales signals are autocorrelated.

# Constraints: plain paragraph only — no headings, bullets, or tables. Refer to drivers
# exclusively by taxonomy category name; never write a raw feature or column name. Bold
# every category name on first use. Report numbers to at most 1 decimal place. Do not
# exceed {max_words} words. Avoid ranking language and superlative flourishes; let the
# numbers sit inside substantive statements about demand.
# """




# -----------------------------------------------------------------------------------------------------------------------------------
# ------------------------------------------------------------- Version 2 -----------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------------------------------

# =============================================================================
# PROMPT PLAYGROUND — edit freely, then re-run the generation cell below.
# =============================================================================

EXPLANATION_SYSTEM_PROMPT = """
You are a senior supply-chain data scientist writing the interpretation paragraph of a
demand-forecasting report in Bangladesh.

You are given SHAP-based global feature attributions from a trained sales-forecasting
model, aggregated to the parent categories of a supply-chain taxonomy.

Ground rules:

- SHAP tells you ASSOCIATION, never causation. Never claim a category "causes" the outcome.
- Use ONLY the numbers provided. Never invent categories, values, mechanisms, or external events.
- `importance` = sum of mean(|SHAP|) over the category -> magnitude of predictive influence.
- Refer to drivers ONLY by their taxonomy category name (e.g. "Operations and Logistics",
  "Sales", "Macroeconomics"). NEVER mention raw feature or column names such as
  `delivery_sales_ratio`, `Sales_lag_7` or `Delivery`.

Voice and style:

- Write like an experienced analyst explaining the situation to another professional,
  not like an AI describing the internal workings of a machine-learning model.
- Start from the BUSINESS DOMAIN and what the attribution pattern suggests for the forecast.
  Category names should normally be the grammatical subject of the sentence.
- Prefer natural constructions such as:
    "Operations and Logistics accounts for 85.4% of the predictive importance..."
    "Sales contributes 13.2% and is associated with downward pressure on the forecast..."
    "Macroeconomic conditions have only a limited role in this forecast..."
- Do NOT use model-centric constructions such as:
    "the model puts weight on...",
    "the model assigns importance to...",
    "the model relies on...",
    "the model focuses on...",
    "the model considers...",
    "the model gives attention to...",
    "the model's explanatory weight...",
    "the model has learned that...",
    "the model is driven by...",
  or close paraphrases.
- Mention "the model" only when technically necessary to clarify that the reported
  relationships are predictive associations rather than causal effects.

- Explain the practical meaning of the attribution pattern without inventing mechanisms.
  Do not infer specific operational problems such as delivery delays, stock shortages,
  capacity constraints, or fulfilment failures unless those concepts are explicitly
  present in the supplied evidence.

- Do NOT narrate a leaderboard. Never use rank-announcing or promotional constructions
  such as:
  "dominates", "by a wide margin", "the next most influential",
  "the next substantive contributor", "the next largest", "second most",
  "comes in second", "ranks third", "the top driver", "followed by",
  "the lion's share", "overwhelmingly", "far and away", "clearly leads",
  "in comparison", or "the remaining categories".

- Never introduce a category by its rank. Introduce the category directly and state what
  its contribution and direction imply for the forecast.

- Weave percentages naturally into sentences that convey an analytical point.
  Avoid sentences whose sole purpose is to report a category and a number.

- Treat very small categories collectively where appropriate rather than creating a
  mechanical sequence of category-percentage statements.

- Keep interpretation proportional to the evidence. Large attribution shares justify
  attention; small attribution shares should not be overinterpreted.

- Vary sentence structure and avoid template-like enumeration
  ("First...", "Second...", "Third...").

- Write dense, natural, decision-oriented prose with no filler, no restatement of the
  task, no headings, no bullet lists, no tables, and no generic AI-style commentary.
"""


EXPLANATION_USER_PROMPT_TEMPLATE = """
## Forecasting target

{target}

## Taxonomy category attributions

{category_table}

## Task

Write ONE natural, professional paragraph of at most {max_words} words interpreting
these attributions for a supply-chain planning audience.

Write from the perspective of the BUSINESS SITUATION rather than from the perspective
of the machine-learning model. Begin directly with the business category carrying the
largest attribution and explain its contribution naturally. Do NOT begin with phrases
such as "the model puts...", "the model assigns...", "the model relies on...",
"the model focuses on...", or "the model's explanatory weight...".

The paragraph should naturally communicate:

- which business domain accounts for most of the predictive importance and the direction
  of its association with the forecast;
- how other materially relevant domains contribute and whether their signals are
  associated with upward or downward forecast movement;
- which domains make only a negligible contribution and therefore warrant less immediate
  attention;
- one practical monitoring implication tied directly to the supplied taxonomy categories;
- a brief closing acknowledgement that the reported relationships are predictive
  associations rather than causes and that historical-sales signals are autocorrelated.

Do not invent a business mechanism that is not contained in the supplied evidence.
For example, do not translate "Operations and Logistics" into specific claims about
delivery delays, inventory shortages, fulfilment failures, or production constraints
unless such information is explicitly provided.

Constraints:

- Plain paragraph only; no headings, bullets, or tables.
- Refer to drivers exclusively by taxonomy category name.
- Never write a raw feature or column name.
- Bold every category name on first use.
- Report numerical values to at most 1 decimal place.
- Do not exceed {max_words} words.
- Avoid ranking language, leaderboard language, and superlative flourishes.
- Avoid describing what "the model thinks", "the model focuses on", or where
  "the model puts its weight".
- Let the numbers support a natural business interpretation rather than becoming
  the structure of the paragraph.
"""


In [ ]:
def format_category_table(df: pd.DataFrame) -> str:
    """
    Aggregate the feature-importance dataframe up to taxonomy parent categories.

    Individual feature names are deliberately NOT exposed to the LLM — only the
    `immediate_parent` categories, their importance share, and their net direction.
    """
    total = df["feature importance"].sum()
    has_signed = "shap_signed" in df.columns

    rows = []
    for cat, g in df.groupby("immediate_parent", dropna=False):
        imp = g["feature importance"].sum()

        # importance-weighted direction vote -> net direction of the category
        up = g.loc[g["direction"].str.lower().eq("positive"), "feature importance"].sum()
        down = g.loc[g["direction"].str.lower().eq("negative"), "feature importance"].sum()
        if up > down:
            net = "raises the forecast"
        elif down > up:
            net = "lowers the forecast"
        else:
            net = "mixed"

        row = {
            "category": cat,
            "n_features": len(g),
            "importance": imp,
            "share": 100 * imp / total,
            "net_direction": net,
            "up_share": 100 * up / imp if imp else 0.0,
        }
        if has_signed:
            row["signed"] = g["shap_signed"].sum()
        rows.append(row)

    agg = pd.DataFrame(rows).sort_values("importance", ascending=False)

    header = "| category | # underlying features | importance | share % | net direction | % of importance pushing up |"
    sep = "|---|---:|---:|---:|---|---:|"
    if has_signed:
        header += " net signed SHAP |"
        sep += "---:|"

    lines = [header, sep]
    for _, r in agg.iterrows():
        line = (f"| {r['category']} | {int(r['n_features'])} | {r['importance']:.1f} | "
                f"{r['share']:.1f} | {r['net_direction']} | {r['up_share']:.1f} |")
        if has_signed:
            line += f" {r['signed']:+.1f} |"
        lines.append(line)
    return "\n".join(lines)


def generate_feature_importance_explanation(
    df: pd.DataFrame,
    model: str,
    *,
    target: str = "Daily product-level sales volume",
    max_words: int = 160,
    temperature: float = 0.2,
    system_prompt: Optional[str] = None,
    user_prompt_template: Optional[str] = None,
    return_prompt: bool = False,
    verbose: bool = False,
):
    """
    Turn a feature-importance dataframe into a short, single-paragraph LLM explanation
    written purely in terms of taxonomy categories (`immediate_parent`).

    Raw feature names are aggregated away before the prompt is built, so the model
    cannot mention columns like `delivery_sales_ratio`.

    Required columns : 'Feature name', 'feature importance', 'direction', 'immediate_parent'
    Optional column  : 'shap_signed' (included in the prompt when present)

    Parameters
    ----------
    model : str
        OpenRouter model slug, e.g. ``"openai/gpt-5.6-luna"`` or
        ``"google/gemini-3.5-flash-lite"``.
    max_words : int
        Hard word budget for the paragraph.
    return_prompt : bool
        If True, returns ``(explanation, rendered_user_prompt)``.
    verbose : bool
        If True, prints the rendered prompt before calling the model.
    """
    required = {"Feature name", "feature importance", "direction", "immediate_parent"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    system_prompt = system_prompt or EXPLANATION_SYSTEM_PROMPT
    user_prompt_template = user_prompt_template or EXPLANATION_USER_PROMPT_TEMPLATE

    user_prompt = user_prompt_template.format(
        target=target,
        category_table=format_category_table(df),
        max_words=max_words,
    )

    if verbose:
        print(user_prompt)
        print("-" * 80)

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        extra_headers=OPENROUTER_HEADERS,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    explanation = (response.choices[0].message.content or "").strip()

    return (explanation, user_prompt) if return_prompt else explanation


In [ ]:
# MODEL = "google/gemini-3.5-flash-lite" # "openai/gpt-5.5" # "openai/gpt-5.6-sol"


explanation, rendered_prompt = generate_feature_importance_explanation(
    feature_impoprtance_for_explanation,
    model=MODEL,
    target="Daily sales volume",
    max_words=100,
    # temperature=0.2,
    return_prompt=True,
)

display(Markdown(explanation))


In [ ]:
explanation_record = {
    "start_date": start_date,
    "end_date": end_date,
    "top_n": top_n,
    "model_name": MODEL,
    "explanation": explanation,
}

output_dir = Path("../../llm_explanations/feature_importance/") 
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"feature_importance_explanation_{start_date}_{end_date}_top_{top_n}_{MODEL.split("/")[-1]}.json"

with open(output_path, "w", encoding="utf-8") as fp:
    json.dump(explanation_record, fp, indent=2, ensure_ascii=False)

print(f"✓ Saved to {output_path}")

In [ ]:
display(Markdown(rendered_prompt))

<br> 

---

<br> <br>

# Generate Explanation from Causality and Event Graph

In [18]:
causality[(causality["cause"] == "exports") | (causality["effect"] == "exports")]

,cause,effect,lag,coefficient,direction
107,exports,NaN,NaN,0.0,none


In [19]:
# feature -> edge_label -> event summary

cols = ["feature", "edge_label", "event_summary"]
event_graph = merged_graph[merged_graph["edge_type"] != "causality"][cols]
event_graph

,feature,edge_label,event_summary
108,inflation_rate,reduces purchasing power,Consumer inflation hits households
109,inflation_rate,raises price levels,Ukraine war raised commodity prices
110,currency_log_return,raises import costs,Taka weakened against dollar
111,exports,slows export growth,Global uncertainty slows exports
112,inflation_rate,raises inflation,Fuel-price adjustments raise inflation
...,...,...,...
571,inflation_rate,raises price levels,Crises raise prices
572,inflation_rate,aims to contain,Proposed budget targets inflation
573,inflation_rate,raises inflation,Consumer prices raise inflation
574,exports,supports economic recovery,"Policy raises exports, aids recovery"


In [20]:
# Search for features that are common in both causality and the event graph
# These are the Macroeconomic features in causality 

causal_features = set(causality["cause"].dropna()) | set(causality["effect"].dropna())
event_graph_macroeconomic_features = set(event_graph[event_graph["feature"].isin(causal_features)]["feature"])
print(event_graph_macroeconomic_features)

{'stock_market_close', 'exports'}


## 3. Recommended explicit version

In [21]:
# from event_graph_cleaner import clean_graph


# ------------------------------------------------------------
# 1. Find variables represented in the causal graph
# ------------------------------------------------------------

causal_features = (
    set(causality["cause"].dropna())
    | set(causality["effect"].dropna())
)


# ------------------------------------------------------------
# 2. Keep only event evidence connected to causal variables
# ------------------------------------------------------------

g = event_graph[
    event_graph["feature"].isin(causal_features)
].copy()


print("Event rows before filtering:", len(event_graph))
print("Event rows after causal filtering:", len(g))

print("\nFeatures retained:")
print(g["feature"].value_counts())


# ------------------------------------------------------------
# 3. Reduce the event graph
# ------------------------------------------------------------

summarized_graph = clean_graph(
    g,

    # Final maximum number of event propositions
    top_n=10,

    # Events with sufficiently high semantic similarity
    # may be grouped together.
    similarity_threshold=0.82,

    # Preserve at least one event from EVERY feature.
    #
    # Therefore stock_market_close cannot disappear simply because
    # exports has many more event rows.
    ensure_each_feature=True,

    # False means every UNIQUE proposition receives equal weight.
    #
    # I recommend this for the main experiment.
    balance_features=False,

    # Keep cluster/provenance information for scientific auditing.
    include_metadata=True,

    verbose=True,
)


display(summarized_graph)

Event rows before filtering: 468
Event rows after causal filtering: 98

Features retained:
feature
exports               92
stock_market_close     6
Name: count, dtype: int64


c:\Users\Iman\OneDrive\Desktop\Supply Chain Study\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4627.32it/s]


Event graph reduction
---------------------
Input rows                  : 98
Distinct features           : 2
After exact deduplication   : 95
Semantic concepts/clusters : 57
Returned rows               : 10
Preserve every feature      : True

Returned rows per feature:
feature
exports               8
stock_market_close    2


,selection_rank,feature,edge_label,event_summary,cluster_unique_events,cluster_total_support,mean_intra_similarity,min_intra_similarity,marginal_coverage_gain,cumulative_semantic_coverage,source_row_ids,cluster_member_uids,semantic_cluster
0,1,exports,increases exports,Export initiatives target products,3,3,0.878102,0.867942,0.676995,0.676995,"[143, 181, 408]","[5, 18, 63]",exports__cluster_18
1,2,stock_market_close,raises market index,Investors drive stock rebound,2,2,0.938079,0.938079,0.051142,0.728137,"[270, 472]","[35, 74]",stock_market_close__cluster_1
2,3,exports,slows export growth,Customer pressures reduce exports,6,6,0.926198,0.869932,0.044000,0.772137,"[174, 251, 296, 297, 395, 457]","[16, 29, 42, 43, 61, 70]",exports__cluster_16
3,4,exports,encourages export growth,Economic zones boost exports,4,4,0.943272,0.928606,0.016439,0.788575,"[291, 362, 443, 455]","[39, 55, 66, 69]",exports__cluster_20
4,5,exports,disrupts exports,Gas crisis affects exports,2,2,0.912957,0.912957,0.009489,0.798064,"[144, 162]","[6, 14]",exports__cluster_17
5,6,exports,facilitates recovery,Export earnings aided recovery,2,2,0.964128,0.964128,0.008921,0.806986,"[178, 260]","[17, 31]",exports__cluster_25
6,7,exports,boosts exports,Agricultural mechanization boosts exports,3,3,0.856790,0.820437,0.007959,0.814945,"[188, 502, 529]","[20, 82, 85]",exports__cluster_0
7,8,exports,facilitates exports,Ports developed to facilitate exports,3,3,0.868956,0.841814,0.007537,0.822482,"[123, 259, 474]","[1, 30, 75]",exports__cluster_7
8,9,exports,reduces exports,Inflation reduces export demand,2,2,0.875822,0.875822,0.006248,0.828730,"[145, 300]","[7, 44]",exports__cluster_5
9,10,stock_market_close,drives market decline,Large-cap securities drove decline,2,2,0.868387,0.868387,0.005557,0.834287,"[200, 292]","[21, 40]",stock_market_close__cluster_0


In [22]:
summarized_graph[
    [
        "feature",
        "edge_label",
        "event_summary",
        "cluster_unique_events",
        "cluster_total_support",
    ]
]

,feature,edge_label,event_summary,cluster_unique_events,cluster_total_support
0,exports,increases exports,Export initiatives target products,3,3
1,stock_market_close,raises market index,Investors drive stock rebound,2,2
2,exports,slows export growth,Customer pressures reduce exports,6,6
3,exports,encourages export growth,Economic zones boost exports,4,4
4,exports,disrupts exports,Gas crisis affects exports,2,2
5,exports,facilitates recovery,Export earnings aided recovery,2,2
6,exports,boosts exports,Agricultural mechanization boosts exports,3,3
7,exports,facilitates exports,Ports developed to facilitate exports,3,3
8,exports,reduces exports,Inflation reduces export demand,2,2
9,stock_market_close,drives market decline,Large-cap securities drove decline,2,2


## 4. DataFrame to feed to the LLM

In [23]:
llm_event_graph = summarized_graph[
    [
        "feature",
        "edge_label",
        "event_summary",
    ]
].copy()

display(llm_event_graph)

,feature,edge_label,event_summary
0,exports,increases exports,Export initiatives target products
1,stock_market_close,raises market index,Investors drive stock rebound
2,exports,slows export growth,Customer pressures reduce exports
3,exports,encourages export growth,Economic zones boost exports
4,exports,disrupts exports,Gas crisis affects exports
5,exports,facilitates recovery,Export earnings aided recovery
6,exports,boosts exports,Agricultural mechanization boosts exports
7,exports,facilitates exports,Ports developed to facilitate exports
8,exports,reduces exports,Inflation reduces export demand
9,stock_market_close,drives market decline,Large-cap securities drove decline


In [24]:
# # Or even easier, tell clean_graph() directly:

# llm_event_graph = clean_graph(
#     g,
#     top_n=10,
#     similarity_threshold=0.82,
#     ensure_each_feature=True,
#     include_metadata=False,
# )

# display(llm_event_graph)

## 5. Converting it to an LLM prompt

I would actually send it as records rather than CSV text:

In [27]:
event_evidence = llm_event_graph.to_dict(
    orient="records"
)

event_evidence

[{'feature': 'exports',
  'edge_label': 'increases exports',
  'event_summary': 'Export initiatives target products'},
 {'feature': 'stock_market_close',
  'edge_label': 'raises market index',
  'event_summary': 'Investors drive stock rebound'},
 {'feature': 'exports',
  'edge_label': 'slows export growth',
  'event_summary': 'Customer pressures reduce exports'},
 {'feature': 'exports',
  'edge_label': 'encourages export growth',
  'event_summary': 'Economic zones boost exports'},
 {'feature': 'exports',
  'edge_label': 'disrupts exports',
  'event_summary': 'Gas crisis affects exports'},
 {'feature': 'exports',
  'edge_label': 'facilitates recovery',
  'event_summary': 'Export earnings aided recovery'},
 {'feature': 'exports',
  'edge_label': 'boosts exports',
  'event_summary': 'Agricultural mechanization boosts exports'},
 {'feature': 'exports',
  'edge_label': 'facilitates exports',
  'event_summary': 'Ports developed to facilitate exports'},
 {'feature': 'exports',
  'edge_label':

In [28]:
import json

event_evidence_text = json.dumps(
    event_evidence,
    indent=2,
    ensure_ascii=False,
)

print(event_evidence_text)

[
  {
    "feature": "exports",
    "edge_label": "increases exports",
    "event_summary": "Export initiatives target products"
  },
  {
    "feature": "stock_market_close",
    "edge_label": "raises market index",
    "event_summary": "Investors drive stock rebound"
  },
  {
    "feature": "exports",
    "edge_label": "slows export growth",
    "event_summary": "Customer pressures reduce exports"
  },
  {
    "feature": "exports",
    "edge_label": "encourages export growth",
    "event_summary": "Economic zones boost exports"
  },
  {
    "feature": "exports",
    "edge_label": "disrupts exports",
    "event_summary": "Gas crisis affects exports"
  },
  {
    "feature": "exports",
    "edge_label": "facilitates recovery",
    "event_summary": "Export earnings aided recovery"
  },
  {
    "feature": "exports",
    "edge_label": "boosts exports",
    "event_summary": "Agricultural mechanization boosts exports"
  },
  {
    "feature": "exports",
    "edge_label": "facilitates exports",

In [ ]:
# taxonomy
# event_graph_macroeconomic_features

In [29]:
taxonomy_level_event_graph_macroeconomic_features = taxonomy[taxonomy["feature_name"].isin(event_graph_macroeconomic_features)| taxonomy["child"].isin(event_graph_macroeconomic_features)]

taxonomy_level_event_graph_macroeconomic_features

# taxonomy_level_event_graph_macroeconomic_features

,feature_name,child,level
57,Macroeconomics,exports,3
61,Macroeconomics,stock_market_close,3


In [ ]:
# taxonomy_level_event_graph_macroeconomic_features = taxonomy[taxonomy["feature_name"].isin(event_graph_macroeconomic_features)| taxonomy["child"].isin(event_graph_macroeconomic_features)]

# taxonomy_level_event_graph_macroeconomic_features

In [30]:
taxonomy_in_causality = taxonomy[
    taxonomy["feature_name"].isin(causal_features)
    | taxonomy["child"].isin(causal_features)
]

taxonomy_in_causality

,feature_name,child,level
10,Operations and Logistics,Delivery,2
11,Operations and Logistics,Factory_Issue,2
12,Operations and Logistics,Production,2
13,Operations and Logistics,Sales,2
15,Operations and Logistics,delivery_sales_ratio,2
16,Operations and Logistics,factory_issue_rate,2
17,Operations and Logistics,inventory_proxy,2
18,Operations and Logistics,production_sales_ratio,2
57,Macroeconomics,exports,3
61,Macroeconomics,stock_market_close,3


In [31]:
def map_causality_to_taxonomy_level(causality: pd.DataFrame, taxonomy: pd.DataFrame) -> pd.DataFrame:
    """
    Map each row's cause and effect to its taxonomy-level feature.

    Both causality["cause"] and causality["effect"] are looked up against
    taxonomy["child"], and the matching taxonomy["feature_name"] is returned
    as the taxonomy-level feature for each side.

    Returns a dataframe with:
        cause, effect, lag, coefficient, direction,
        cause_taxonomy_level_feature, effect_taxonomy_level_feature
    """
    child_to_feature = taxonomy.set_index("child")["feature_name"]

    result = causality[["cause", "effect", "lag", "coefficient", "direction"]].copy()
    result["cause_taxonomy_level_feature"] = result["cause"].map(child_to_feature)
    result["effect_taxonomy_level_feature"] = result["effect"].map(child_to_feature)

    return result


causality_taxonomy_level = map_causality_to_taxonomy_level(causality, taxonomy)
causality_taxonomy_level

,cause,effect,lag,coefficient,direction,cause_taxonomy_level_feature,effect_taxonomy_level_feature
0,Sales_lag_28,Production,0.0,0.932276,positive,Sales,Operations and Logistics
1,delivery_sales_ratio,production_sales_ratio,0.0,0.909791,positive,Operations and Logistics,Operations and Logistics
2,Sales_lag_28,Sales_lag_3,0.0,0.850840,positive,Sales,Sales
3,Sales_lag_28,Sales_lag_7,0.0,0.834155,positive,Sales,Sales
4,Sales_lag_7,Sales_lag_21,0.0,0.747857,positive,Sales,Sales
...,...,...,...,...,...,...,...
103,Product_SOS001L12P,Delivery,1.0,-0.055638,negative,Products,Operations and Logistics
104,Sales_lag_14,Sales_lag_1,1.0,-0.055528,negative,Sales,Sales
105,Product_SOS001L12P,Factory_Issue,1.0,-0.055279,negative,Products,Operations and Logistics
106,Product_SOS001L12P,Sales,1.0,-0.050122,negative,Products,Operations and Logistics


In [32]:
def drop_same_taxonomy_level_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove rows where cause and effect map to the same taxonomy-level feature.

    Rows where either side is unmapped (NaN) are kept, since they are not equal.
    """
    mask = df["cause_taxonomy_level_feature"] != df["effect_taxonomy_level_feature"]
    return df[mask].reset_index(drop=True)


causality_taxonomy_level_filtered = drop_same_taxonomy_level_rows(causality_taxonomy_level)

print(f"Before: {len(causality_taxonomy_level)} rows")
print(f"After:  {len(causality_taxonomy_level_filtered)} rows")
causality_taxonomy_level_filtered

Before: 108 rows
After:  68 rows


,cause,effect,lag,coefficient,direction,cause_taxonomy_level_feature,effect_taxonomy_level_feature
0,Sales_lag_28,Production,0.0,0.932276,positive,Sales,Operations and Logistics
1,Production,Sales_lag_1,0.0,0.636916,positive,Operations and Logistics,Sales
2,Production,Sales_lag_2,0.0,0.636178,positive,Operations and Logistics,Sales
3,Temperature,stock_market_close,0.0,0.590163,positive,Weather,Macroeconomics
4,Temperature,stock_market_close,1.0,-0.580860,negative,Weather,Macroeconomics
...,...,...,...,...,...,...,...
63,Product_SOS001L12P,Sales_lag_7,0.0,-0.061647,negative,Products,Sales
64,Product_SOS001L12P,Delivery,1.0,-0.055638,negative,Products,Operations and Logistics
65,Product_SOS001L12P,Factory_Issue,1.0,-0.055279,negative,Products,Operations and Logistics
66,Product_SOS001L12P,Sales,1.0,-0.050122,negative,Products,Operations and Logistics


In [33]:
causality_taxonomy_level_filtered.iloc[[3, 4]]

,cause,effect,lag,coefficient,direction,cause_taxonomy_level_feature,effect_taxonomy_level_feature
3,Temperature,stock_market_close,0.0,0.590163,positive,Weather,Macroeconomics
4,Temperature,stock_market_close,1.0,-0.580860,negative,Weather,Macroeconomics


In [34]:
def drop_duplicate_taxonomy_level_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only the first row for each unique
    (cause_taxonomy_level_feature, effect_taxonomy_level_feature) pair.
    """
    return df.drop_duplicates(
        subset=["cause_taxonomy_level_feature", "effect_taxonomy_level_feature"],
        keep="first",
    ).reset_index(drop=True)


causality_taxonomy_level_deduped = drop_duplicate_taxonomy_level_pairs(causality_taxonomy_level_filtered)

print(f"Before: {len(causality_taxonomy_level_filtered)} rows")
print(f"After:  {len(causality_taxonomy_level_deduped)} rows")
causality_taxonomy_level_deduped

Before: 68 rows
After:  11 rows


,cause,effect,lag,coefficient,direction,cause_taxonomy_level_feature,effect_taxonomy_level_feature
0,Sales_lag_28,Production,0.0,0.932276,positive,Sales,Operations and Logistics
1,Production,Sales_lag_1,0.0,0.636916,positive,Operations and Logistics,Sales
2,Temperature,stock_market_close,0.0,0.590163,positive,Weather,Macroeconomics
3,Product_SOS001L12P,inventory_proxy,1.0,-0.562555,negative,Products,Operations and Logistics
4,Product_SOS001L12P,Sales_lag_28,0.0,0.335067,positive,Products,Sales
5,Temperature,Sales_lag_2,0.0,0.220901,positive,Weather,Sales
6,stock_market_close,Sales_lag_2,0.0,-0.186389,negative,Macroeconomics,Sales
7,delivery_sales_ratio,Product_SOS001L12P,0.0,0.184633,positive,Operations and Logistics,Products
8,Temperature,Production,0.0,0.136044,positive,Weather,Operations and Logistics
9,production_sales_ratio,stock_market_close,1.0,-0.103674,negative,Operations and Logistics,Macroeconomics


In [35]:
# cols = ["cause_taxonomy_level_feature", "effect_taxonomy_level_feature"]
display(causality_taxonomy_level_deduped[["cause_taxonomy_level_feature", "effect_taxonomy_level_feature"]])
# print(event_evidence_text)

,cause_taxonomy_level_feature,effect_taxonomy_level_feature
0,Sales,Operations and Logistics
1,Operations and Logistics,Sales
2,Weather,Macroeconomics
3,Products,Operations and Logistics
4,Products,Sales
5,Weather,Sales
6,Macroeconomics,Sales
7,Operations and Logistics,Products
8,Weather,Operations and Logistics
9,Operations and Logistics,Macroeconomics


In [36]:
print(event_evidence_text)

[
  {
    "feature": "exports",
    "edge_label": "increases exports",
    "event_summary": "Export initiatives target products"
  },
  {
    "feature": "stock_market_close",
    "edge_label": "raises market index",
    "event_summary": "Investors drive stock rebound"
  },
  {
    "feature": "exports",
    "edge_label": "slows export growth",
    "event_summary": "Customer pressures reduce exports"
  },
  {
    "feature": "exports",
    "edge_label": "encourages export growth",
    "event_summary": "Economic zones boost exports"
  },
  {
    "feature": "exports",
    "edge_label": "disrupts exports",
    "event_summary": "Gas crisis affects exports"
  },
  {
    "feature": "exports",
    "edge_label": "facilitates recovery",
    "event_summary": "Export earnings aided recovery"
  },
  {
    "feature": "exports",
    "edge_label": "boosts exports",
    "event_summary": "Agricultural mechanization boosts exports"
  },
  {
    "feature": "exports",
    "edge_label": "facilitates exports",

In [37]:
# =============================================================================
# CAUSAL + EVENT NARRATIVE PROMPTS — edit freely, then re-run the cell below.
# =============================================================================

CAUSAL_EVENT_SYSTEM_PROMPT = """\
You are a supply-chain analyst writing a short briefing for a non-technical business
audience (demand planners, sourcing managers, commercial leads) about sales demand in
Bangladesh.

You are given two independent pieces of evidence:
1. CAUSAL STRUCTURE — directed cause -> effect links discovered between broad
   business domains of a supply-chain taxonomy (e.g. "Weather" -> "Macroeconomics").
2. NEWS EVENT EVIDENCE — real-world events extracted from news articles, each linked to
   the business domain it affects and the direction of that effect.

Non-negotiable rules:
- NEVER write raw feature, column, or variable names. Forbidden examples:
  `stock_market_close`, `Sales_lag_7`, `delivery_sales_ratio`, `production_sales_ratio`,
  `Product_SOS001L12P`, `inventory_proxy`, `Factory_Issue`, `exports`, `Temperature`.
  Use plain business language instead: "stock market performance", "recent sales
  momentum", "delivery-to-sales throughput", "production efficiency", "the product
  portfolio", "inventory cover", "factory disruptions", "export activity",
  "temperature and weather".
- Use ONLY the evidence provided. Never invent links, events, numbers, or dates.
- The causal links come from statistical causal discovery on observational data. Treat
  them as PLAUSIBLE, DATA-SUPPORTED influence, not proven mechanism. Hedge appropriately
  ("appears to influence", "tends to feed through into"), and never overstate certainty.
- Never quote the numeric coefficients or lag values; translate them into qualitative
  strength ("strong", "moderate", "weak") and timing ("same-day", "with about a day's
  delay") only if that information is supplied.
- Where a news event sits on the same business domain as a causal link, connect the two
  explicitly — that combination is the most valuable insight for the reader.
- If a causal link has no supporting event, do not fabricate one; simply present it as a
  structural relationship visible in the data.
- Write clear, confident, jargon-free business prose as ONE single paragraph. No headings,
  no bullet lists, no markdown tables, no restating the task, no filler openers like
  "In summary" or "This report".
- Brevity is a hard requirement. The word limit given in the task is absolute; every
  sentence must earn its place.
"""

CAUSAL_EVENT_USER_PROMPT_TEMPLATE = """\
## What is being explained
{target}

## Evidence A — causal links between business domains
Each line is a directed relationship discovered in the historical data: the first domain
appears to influence the second.

{causal_block}

## Evidence B — real-world events from news, by affected business domain
Each entry names a business domain, the direction of the effect on it, and a one-line
summary of the underlying news event.

{event_block}

## Task
Write ONE single continuous paragraph of flowing prose, at most {max_words} words, that
covers the following in this order without ever announcing the structure:
1. the single most consequential chain in Evidence A, explained in plain business terms
   so the reader sees how influence travels along it;
2. why this is happening right now, using Evidence B — group the events by the domain they
   affect, note where events push in opposing directions, and state which direction
   currently dominates; tie at least two events back to a link from Evidence A so the
   reader sees the mechanism rather than a list;
3. {n_actions} concrete monitoring or planning actions that follow directly from that
   evidence, each anchored to a named business domain;
4. a closing clause noting that these relationships are inferred from historical data and
   news signals and should be treated as informed guidance rather than certainty.

Hard constraints:
- ONE plain paragraph only. No headings, no bullets, no tables, no numbered lists, no
  line breaks, no section labels in the output.
- HARD LIMIT: {max_words} words maximum. Count your words and stay under the limit; being
  well under it is better than exceeding it. Cut detail rather than run over.
- Bold each business domain the first time you mention it.
- ZERO raw feature or column names anywhere in the output.
"""


In [38]:
import json
import re

# Raw column/feature name -> user-friendly business phrase.
# Anything not listed falls back to a generic de-snake-cased label.
FEATURE_ALIASES = {
    "exports": "export activity",
    "stock_market_close": "stock market performance",
    "inflation": "consumer price inflation",
    "temperature": "temperature and weather conditions",
    "rainfall": "rainfall levels",
    "precipitation": "rainfall levels",
    "humidity": "humidity levels",
    "sales": "recent sales momentum",
    "production": "factory production output",
    "delivery": "outbound deliveries",
    "factory_issue": "factory disruptions",
    "inventory_proxy": "inventory cover",
    "delivery_sales_ratio": "delivery-to-sales throughput",
    "production_sales_ratio": "production-to-sales efficiency",
}


def humanize_feature(name: Any) -> str:
    """Convert a raw feature/column name into a user-friendly business phrase."""
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return "an unclassified area"

    text = str(name).strip()
    key = text.lower()

    if key in FEATURE_ALIASES:
        return FEATURE_ALIASES[key]

    # Strip lag suffixes such as `Sales_lag_28` -> `sales`
    base = re.sub(r"_lag_\d+$", "", text)
    if base.lower() in FEATURE_ALIASES:
        return FEATURE_ALIASES[base.lower()]

    # Product SKU codes -> generic phrase
    if re.match(r"^product[_\-]", base, flags=re.IGNORECASE):
        return "the product portfolio"

    # Generic fallback: de-snake-case, keep taxonomy names ("Operations and Logistics") intact
    return base.replace("_", " ").strip() or "an unclassified area"


def format_causal_block(df: pd.DataFrame) -> str:
    """
    Render taxonomy-level causal pairs as user-friendly directed statements.

    Expects the two columns 'cause_taxonomy_level_feature' and
    'effect_taxonomy_level_feature'. Rows with a missing effect are reported as
    context-only domains rather than as links.
    """
    required = {"cause_taxonomy_level_feature", "effect_taxonomy_level_feature"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    links, context_only = [], []
    for _, r in df.iterrows():
        cause = r["cause_taxonomy_level_feature"]
        effect = r["effect_taxonomy_level_feature"]

        if pd.isna(cause):
            continue
        if pd.isna(effect):
            context_only.append(humanize_feature(cause))
            continue
        links.append(f"- {humanize_feature(cause)} -> {humanize_feature(effect)}")

    if not links and not context_only:
        return "- (no causal links available)"

    lines = links or ["- (no directed links available)"]
    if context_only:
        seen = list(dict.fromkeys(context_only))
        lines.append(
            "- Additional domains present in the data with no outgoing link: "
            + ", ".join(seen)
        )
    return "\n".join(lines)


def format_event_block(event_evidence_text: str) -> str:
    """
    Normalize the event-graph evidence into user-friendly lines grouped by domain.

    Accepts the JSON string produced from `llm_event_graph.to_dict(orient="records")`.
    Falls back to passing the text through unchanged if it is not valid JSON.
    """
    try:
        records = json.loads(event_evidence_text)
    except (TypeError, ValueError):
        return event_evidence_text.strip() or "- (no event evidence available)"

    if isinstance(records, dict):
        records = [records]
    if not records:
        return "- (no event evidence available)"

    grouped: dict[str, list[str]] = {}
    for rec in records:
        domain = humanize_feature(rec.get("feature"))
        effect = str(rec.get("edge_label", "")).strip() or "affects it"
        summary = str(rec.get("event_summary", "")).strip() or "(no summary)"
        grouped.setdefault(domain, []).append(f"    - {effect}: {summary}")

    lines = []
    for domain, items in grouped.items():
        lines.append(f"- Business domain: {domain} ({len(items)} event(s))")
        lines.extend(items)
    return "\n".join(lines)


def generate_causal_event_explanation(
    causal_pairs: pd.DataFrame,
    event_evidence_text: str,
    model: str,
    *,
    target: str = "Daily product-level sales demand in Bangladesh",
    max_words: int = 250,
    n_actions: int = 3,
    temperature: float = 0.2,
    max_revisions: int = 2,
    system_prompt: Optional[str] = None,
    user_prompt_template: Optional[str] = None,
    return_prompt: bool = False,
    verbose: bool = False,
):
    """
    Generate a user-friendly LLM explanation from taxonomy-level causal links plus
    news-derived event evidence.

    Raw feature names are translated into business language *before* the prompt is
    built, so the model never sees strings like `stock_market_close`.

    Parameters
    ----------
    causal_pairs : pd.DataFrame
        Must contain 'cause_taxonomy_level_feature' and
        'effect_taxonomy_level_feature', e.g.
        ``causality_taxonomy_level_deduped[["cause_taxonomy_level_feature",
        "effect_taxonomy_level_feature"]]``.
    event_evidence_text : str
        JSON string of event-graph records (feature / edge_label / event_summary).
    model : str
        OpenRouter model slug, e.g. ``"google/gemini-3.5-flash-lite"``.
    max_words : int
        Hard word budget. The output is verified and, if over budget, the model is asked
        to compress it (up to `max_revisions` times).
    return_prompt : bool
        If True, returns ``(explanation, rendered_user_prompt)``.
    """
    system_prompt = system_prompt or CAUSAL_EVENT_SYSTEM_PROMPT
    user_prompt_template = user_prompt_template or CAUSAL_EVENT_USER_PROMPT_TEMPLATE

    user_prompt = user_prompt_template.format(
        target=target,
        causal_block=format_causal_block(causal_pairs),
        event_block=format_event_block(event_evidence_text),
        max_words=max_words,
        n_actions=n_actions,
    )

    if verbose:
        print(user_prompt)
        print("-" * 80)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    def _call(msgs) -> str:
        response = client.chat.completions.create(
            model=model,
            temperature=temperature,
            extra_headers=OPENROUTER_HEADERS,
            messages=msgs,
        )
        return (response.choices[0].message.content or "").strip()

    explanation = _call(messages)

    # Enforce the hard word budget: ask the model to compress if it overran.
    for attempt in range(max_revisions):
        n_words = len(explanation.split())
        if n_words <= max_words:
            break
        if verbose:
            print(f"[revision {attempt + 1}] {n_words} words > {max_words}; compressing...")
        messages = messages + [
            {"role": "assistant", "content": explanation},
            {
                "role": "user",
                "content": (
                    f"That answer is {n_words} words, which exceeds the hard limit of "
                    f"{max_words} words. Rewrite it as ONE paragraph of at most "
                    f"{max_words} words. Keep the most decision-relevant content and cut "
                    f"the rest. Obey every original constraint: one plain paragraph, no "
                    f"headings or bullets, no raw feature or column names, bold each "
                    f"business domain on first mention. Return only the rewritten "
                    f"paragraph."
                ),
            },
        ]
        explanation = _call(messages)

    if verbose:
        print(f"Final word count: {len(explanation.split())} (limit {max_words})")

    return (explanation, user_prompt) if return_prompt else explanation


In [61]:
causality_taxonomy_level_deduped

,cause,effect,lag,coefficient,direction,cause_taxonomy_level_feature,effect_taxonomy_level_feature
0,Sales_lag_28,Production,0.0,0.932276,positive,Sales,Operations and Logistics
1,Production,Sales_lag_1,0.0,0.636916,positive,Operations and Logistics,Sales
2,Temperature,stock_market_close,0.0,0.590163,positive,Weather,Macroeconomics
3,Product_SOS001L12P,inventory_proxy,1.0,-0.562555,negative,Products,Operations and Logistics
4,Product_SOS001L12P,Sales_lag_28,0.0,0.335067,positive,Products,Sales
5,Temperature,Sales_lag_2,0.0,0.220901,positive,Weather,Sales
6,stock_market_close,Sales_lag_2,0.0,-0.186389,negative,Macroeconomics,Sales
7,delivery_sales_ratio,Product_SOS001L12P,0.0,0.184633,positive,Operations and Logistics,Products
8,Temperature,Production,0.0,0.136044,positive,Weather,Operations and Logistics
9,production_sales_ratio,stock_market_close,1.0,-0.103674,negative,Operations and Logistics,Macroeconomics


In [63]:
print(event_evidence_text)

[
  {
    "feature": "exports",
    "edge_label": "increases exports",
    "event_summary": "Export initiatives target products"
  },
  {
    "feature": "stock_market_close",
    "edge_label": "raises market index",
    "event_summary": "Investors drive stock rebound"
  },
  {
    "feature": "exports",
    "edge_label": "slows export growth",
    "event_summary": "Customer pressures reduce exports"
  },
  {
    "feature": "exports",
    "edge_label": "encourages export growth",
    "event_summary": "Economic zones boost exports"
  },
  {
    "feature": "exports",
    "edge_label": "disrupts exports",
    "event_summary": "Gas crisis affects exports"
  },
  {
    "feature": "exports",
    "edge_label": "facilitates recovery",
    "event_summary": "Export earnings aided recovery"
  },
  {
    "feature": "exports",
    "edge_label": "boosts exports",
    "event_summary": "Agricultural mechanization boosts exports"
  },
  {
    "feature": "exports",
    "edge_label": "facilitates exports",

In [59]:

causal_event_explanation, causal_event_prompt = generate_causal_event_explanation(
    causality_taxonomy_level_deduped[
        ["cause_taxonomy_level_feature", "effect_taxonomy_level_feature"]
    ],
    event_evidence_text,
    model=MODEL,
    target="Daily product-level sales demand in Bangladesh",
    max_words=100,
    # temperature=0.2,
    return_prompt=True,
    verbose=False,
)

display(Markdown(causal_event_explanation))
print(f"\nWord count: {len(causal_event_explanation.split())} / 250")

Recent sales momentum and **Operations and Logistics** form a reinforcing cycle shaping demand. Currently, **export activity** faces mixed pressures—gas crises and inflation dampen shipments, while port development and economic zones provide support, with headwinds slightly dominating. This export weakness feeds into **Operations and Logistics** efficiency, constraining sales momentum. Monitor **Weather** patterns as they cascade into **Macroeconomics** and sales; track **export activity** recovery signals from infrastructure gains; adjust **Products** mix as logistics tightens. These relationships emerge from historical data and should guide rather than dictate strategy.


Word count: 85 / 250


In [60]:
explanation_record = {
    "start_date": start_date,
    "end_date": end_date,
    "top_n": top_n,
    "model_name": MODEL,
    "explanation": causal_event_explanation,
}

output_dir = Path("../../llm_explanations/causal_event_graph/") 
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"causal_event_graph_{start_date}_{end_date}_top_{top_n}_{MODEL.split("/")[-1]}.json"

with open(output_path, "w", encoding="utf-8") as fp:
    json.dump(explanation_record, fp, indent=2, ensure_ascii=False)

print(f"✓ Saved to {output_path}")

✓ Saved to ..\..\llm_explanations\causal_event_graph\causal_event_graph_2023-07-11_2023-07-17_top_20_claude-haiku-4.5.json


<br> <br> <br>

---

<br> <br>


In [ ]:
display(Markdown(f"## Explanation based on Feature Importace \n{explanation}\n\n\n"))
display(Markdown(f"## Explanation based on Causal and Event Graph \n{causal_event_explanation}"))